# Tutorial 08 - Translation with a Sequence to Sequence Network and Attention

The goal of this tutorial is to build a neural network model to translate from French to English. 

In [ ]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import string
import re
import random
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
%matplotlib inline
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np
import time
import math
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import List
from torch.nn.utils.rnn import pad_sequence
from IPython.display import clear_output
from einops import rearrange

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Processing

In [ ]:
SOS_token = 0
EOS_token = 1
PAD_token = 2

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS", 2: "PAD"}
        self.n_words = 3  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [ ]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )
# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

In [ ]:
def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    # Read the file and split into lines
    lines = open('%s-%s.txt' % (lang1, lang2), encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs, make Lang instances
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

In [ ]:
MAX_LENGTH = 10
eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)
def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [ ]:
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs


input_lang, output_lang, pairs = prepareData('eng', 'fra', True)
print(random.choice(pairs))

In [ ]:
print(random.choice(pairs))

In [ ]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

In [ ]:
class LangPairsDataset(Dataset):
    def __init__(self, pairs: List):
        super(LangPairsDataset, self).__init__()

        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index: int):
        return tensorsFromPair(self.pairs[index])


In [ ]:
lang_pairs_dataset = LangPairsDataset(pairs)
print(lang_pairs_dataset[20])
print(lang_pairs_dataset.pairs[20])

In [ ]:
def get_dataloader(dataset, num_iters, batch_size):
    sampler = torch.utils.data.RandomSampler(
        dataset,
        replacement=True,
        num_samples=num_iters * batch_size,
        generator=torch.Generator().manual_seed(42))
    return DataLoader(dataset=dataset,
                      batch_size=batch_size,
                      collate_fn=PairsCollate(),
                      sampler=sampler)


class PairsCollate(object):
    def __init__(self):
        super(PairsCollate, self).__init__()

    @staticmethod
    def __call__(items):
        items1 = [s[0] for s in items]
        items2 = [s[1] for s in items]
        batch1 = pad_sequence(items1, batch_first=True, padding_value=PAD_token)
        batch2 = pad_sequence(items2, batch_first=True, padding_value=PAD_token)

        return batch1.squeeze(-1), batch2.squeeze(-1)

In [ ]:
len(lang_pairs_dataset)

In [ ]:
next(iter(get_dataloader(lang_pairs_dataset, num_iters=100, batch_size=256)))[0].shape

## Model

Now you are to implement an encoder and two decoder variants to map a sequence of tokens in one language to a sequence of tokens in another language.

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        #Embedding layer
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, input, hidden=None):
        """
        :param input: [b l]
        :param hidden: [n b h]
        :return: [b l h], [1 b h]
        """
        
        ### YOUR CODE HERE ###

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        
        #Embedding layer
        self.embedding = nn.Embedding(output_size, hidden_size)
        #RNN layer
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        #Projection layer
        self.out = nn.Linear(hidden_size, output_size)
    
    def forward(self, input, encoder_outputs, hidden=None):
        """
        :param input: [b l]
        :param encoder_outputs: [b k h]
        :param hidden: [n b h]
        :return: [b l v], [1 b h]
        """
        
        ### YOUR CODE HERE ###

In [ ]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size

        #Embedding layer
        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        
        #Attention layer to learn the attention weights
        self.q = nn.Linear(self.hidden_size, self.hidden_size)
        self.k = nn.Linear(self.hidden_size, self.hidden_size)
        self.v = nn.Linear(self.hidden_size, self.hidden_size)
        
        # Layer to combine the input and the context vector learned through attention
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        
        #GRU layer
        self.gru = nn.GRU(self.hidden_size, self.hidden_size, batch_first=True)
        
        #Projection layer
        self.out = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input, encoder_outputs, hidden=None):
        """
        :param input: [b l]
        :param encoder_outputs: [b k h]
        :param hidden: [n b h]
        :return: [b l v], [1 b h], [b l k]
        """
        
        ### YOUR CODE HERE ###

## Training

Implement the training code for one batch. Optionally implement the branch without the teacher forcing logic, where you sequentially decode the sentence token by token by feeding the predictions of the network to its input.

In [ ]:
def train_batch(source, target, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, teacher_forcing_ratio=1.0):
    """
    :param source: [b l]
    :param target: [b k]
    """

    ### YOUR CODE HERE ###

In [ ]:
def train(encoder, decoder, pairs, n_iters, batch_size, plot_every=100, learning_rate=0.01):
    encoder.train()
    decoder.train()
    
    plot_losses = []
    plot_loss_total = 0  # Reset every plot_every
    
    # Define optimizers
    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)

    # Define dataloaders
    dataset = LangPairsDataset(pairs)
    dataloader = get_dataloader(dataset, n_iters, batch_size)
    
    # Define the loss
    criterion = nn.CrossEntropyLoss()

    for i, (source, target) in enumerate(dataloader):
        #train and compute loss
        loss = train_batch(source, target, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        
        plot_loss_total += loss
        
        if (i + 1) % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

            clear_output(wait=True)
            x = np.arange(0, len(plot_losses) * plot_every, plot_every)
            plt.plot(x, plot_losses)
            plt.show()

In [ ]:
%matplotlib inline

hidden_size = 256
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)
train(encoder, decoder, pairs, 30000, 32, learning_rate=0.1, plot_every=100)

In [ ]:
%matplotlib inline

hidden_size = 256
attn_encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
attn_decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)
train(attn_encoder, attn_decoder, pairs, 30000, 32, learning_rate=0.1, plot_every=100)

## Evaluation

In [ ]:
def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH):
    """
    :param sentence: [l] sequence of words ids
    :return: list of decoded word ids, attention map
    """
    
    ### YOUR CODE HERE ###

In [ ]:
def evaluateRandomly(encoder, decoder, n=3):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        t, _ = tensorsFromPair(pair)
        output_words, attentions = evaluate(encoder, decoder, t.squeeze(1), max_length=20)
        output_words = [output_lang.index2word[w] for w in output_words if w != EOS_token and w != PAD_token]
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [ ]:
evaluateRandomly(encoder, decoder)

In [ ]:
evaluateRandomly(attn_encoder, attn_decoder)

In [ ]:
def showAttention(input_sentence, output_words, attentions):
    # Set up figure with colorbar
    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(attentions.numpy(), cmap='bone')
    fig.colorbar(cax)

    # Set up axes
    ax.set_xticklabels([''] + input_sentence.split(' ') +
                       ['<EOS>'], rotation=90)
    ax.set_yticklabels([''] +  output_words)

    # Show label at every tick
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    plt.show()

def evaluateAndShowAttention(input_sentence):
    t, _ = tensorsFromPair((input_sentence, "i m"))
    output_words, attentions = evaluate(
        attn_encoder, attn_decoder, t.squeeze(1))
    output_words = [output_lang.index2word[w] for w in output_words if w != EOS_token and w != PAD_token]
    print('input =', input_sentence)
    print('output =', ' '.join(output_words))
    showAttention(input_sentence, output_words, attentions.cpu())


In [ ]:
evaluateAndShowAttention("elle a cinq ans de moins que moi .")

In [ ]:
evaluateAndShowAttention("c est un jeune directeur plein de talent .")